# Quick Test — Groundwater-Level Spatiotemporal GNN

This notebook performs a rapid inference check using:

```text
quick_test_assets/
├── quick_test_data.pt
├── model_state_dict.pth
├── manifest.json
└── target_scaler.joblib   # optional
```

The quick-test samples come from the **held-out experimental test split**.

No model training or full preprocessing is performed here. The objective is to verify that the released model checkpoint can reproduce predictions on released held-out test graphs.

If `target_scaler.joblib` is present, RMSE/MAE are also reported in groundwater-head units. Otherwise, the notebook reports normalized RMSE/MAE plus R² and NSE.

In [ ]:
from pathlib import Path
import json
import random
import platform
import sys

import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path(".")
ASSET_DIR = ROOT / "quick_test_assets"

QUICK_DATA_FILE = ASSET_DIR / "quick_test_data.pt"
MODEL_FILE = ASSET_DIR / "model_state_dict.pth"
SCALER_FILE = ASSET_DIR / "target_scaler.joblib"
MANIFEST_FILE = ASSET_DIR / "manifest.json"
REFERENCE_FILE = ASSET_DIR / "reference_metrics.json"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

## Model definition

In [ ]:
import random
import os
import math
from collections import deque
from torch_geometric.nn import GATv2Conv # 使用 GATv2 更稳定
from torch_geometric.utils import remove_self_loops, add_self_loops
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, TransformerConv
from torch_geometric.utils import softmax
import torch
import torch.nn as nn

class PhysicsAdaptationLayer(nn.Module):
    """
    简化版物理适配层：不做复杂的动态调节，直接拼接特征。
    仅仅负责将原始的 7 维特征整理好，供 GNN 使用。
    """
    def __init__(self, initial_k_factor=None, initial_grad_factor=None):
        super().__init__()
        # 用户要求移除动态调节因子，因此这里不再初始化 nn.Parameter
        # 我们可以保留一个BatchNorm来确保数值稳定性，这不算复杂的逻辑
        self.grad_norm = nn.BatchNorm1d(1)
        self.k_norm = nn.BatchNorm1d(1)
        
    def forward(self, edge_attr_raw):
        """
        Args:
            edge_attr_raw: [E, 7]
            顺序对应 preprocess 中的拼接顺序：
            0: landuse_w
            1: pumping_effect (预处理已经计算好了衰减)
            2: irrigation_effect (预处理已经计算好了衰减)
            3: river_effect (预处理已经计算好了衰减)
            4: elevation_gradient
            5: effective_k_log
            6: norm_dist
        """
        # 1. 提取特征
        landuse_w = edge_attr_raw[:, 0:1]
        pumping_effect = edge_attr_raw[:, 1:2]
        irrigation_effect = edge_attr_raw[:, 2:3]
        river_effect = edge_attr_raw[:, 3:4]
        
        raw_grad = edge_attr_raw[:, 4:5]
        raw_k_log = edge_attr_raw[:, 5:6]
        norm_dist = edge_attr_raw[:, 6:7]

        # 2. 简单归一化物理属性 (可选，但推荐保留以加速收敛)
        norm_grad = self.grad_norm(raw_grad)
        norm_k_log = self.k_norm(raw_k_log)

        # 3. 直接拼接
        # 我们不再计算 grad_modulator = 1 + tanh(...) 这种复杂逻辑
        # 而是相信预处理阶段计算的 pumping_effect 已经包含了物理距离信息
        # GNN 的权重矩阵会自动学习如何结合这些特征
        final_edge_attr = torch.cat([
            landuse_w,          # 0
            pumping_effect,     # 1 (直接使用预处理算好的)
            irrigation_effect,  # 2 (直接使用预处理算好的)
            river_effect,       # 3 (直接使用预处理算好的)
            norm_grad,          # 4
            norm_k_log,         # 5
            norm_dist           # 6
        ], dim=1)

        return final_edge_attr
# =======================================================
# 新增模块: GNN-NHiTS Block (将GNN作为N-HiTS的核心)
# =======================================================
class GNN_NHiTS_Block(nn.Module):
    """
    一个N-HiTS块，但其核心处理逻辑由我们的GNN Processor驱动。
    它接收一个时间序列输入 (backcast)，并输出一个未来预测 (forecast)
    和一个传递给下一块的残差 backcast。
    """
    def __init__(self,
                 hidden_dim,
                 edge_dim,
                 backcast_length,
                 forecast_length,
                 static_encoder,
                 dynamic_encoder,
                 history_proj,
                 spatial_encoder,
                 fusion_layer,
                 dropout=0.1):
        super().__init__()
        self.backcast_length = backcast_length
        self.forecast_length = forecast_length

        # === 共享的编码器组件 (从主模型传入) ===
        self.static_encoder = static_encoder
        self.dynamic_encoder = dynamic_encoder
        self.history_proj = history_proj
        self.spatial_encoder = spatial_encoder
        self.fusion_layer = fusion_layer

        # === 每个块拥有自己独立的核心处理器 ===
        # 这允许每个块学习不同频率的时空模式
        self.processor = ProcessorModule(hidden_dim, edge_dim, num_gdo_layers=3, dropout=dropout)

        # === 预测头 (MLPs) ===
        # Backcast 头: 从最终状态生成对输入序列的重建/残差
        self.backcast_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, backcast_length)
        )
        # Forecast 头: 从最终状态生成对未来序列的预测
        self.forecast_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, forecast_length)
        )

    def forward(self, backcast_input, data):
        """
        backcast_input: (num_nodes, backcast_length) - 来自上一块的输入序列
        data: 原始的图数据对象，用于获取静态/动态特征和图结构
        """
        edge_index, edge_attr = data.edge_index, data.edge_attr

        # --- 准备静态特征 (只需一次) ---
        static_embed = self.static_encoder(data.x_static)
        spatial_embed = self.spatial_encoder(data.coords)

        # --- GNN编码器循环: 处理输入的 backcast_input 序列 ---
        self.processor.reset_states()
        for t in range(self.backcast_length):
            # 获取当前时间步的动态/历史特征
            dynamic_embed_t = self.dynamic_encoder(
                data.x_dynamic_encoder[:, t, :],
                data.x_time_encoder[:, t, :]
            )
            # 关键：历史输入来自 backcast_input
            history_embed_t = self.history_proj(backcast_input[:, t].unsqueeze(-1))

            # 融合特征并由Processor处理
            combined_embed_t = torch.cat([static_embed, dynamic_embed_t, history_embed_t, spatial_embed], dim=-1)
            x_t = self.fusion_layer(combined_embed_t)
            node_state, edge_attr = self.processor(x_t, edge_index, edge_attr, use_stored_state=True)

        # --- 预测 ---
        # 编码器循环结束后，`node_state` 包含了整个输入序列的时空摘要

        # 1. 生成 Backcast
        backcast_output = self.backcast_head(node_state)

        # 2. 生成 Forecast
        forecast_output = self.forecast_head(node_state)

        return backcast_output, forecast_output

"""
本文件在你现有实现基础上，落地：
- 方案 B（频率控制）：Processor 的前两层关闭边更新，仅最后一层更新边；Encoder 的两层 GDO 里只第一层更新边。
- 方案 A（门控）：凡是“保留边更新”的 GDOBlock，在边的残差更新上加入可学习门控 edge_gate（初值 -1 => sigmoid≈0.27）。

你可以直接替换原文件使用；类名与外部接口保持不变：
- SpatialEncoding / EdgeEnhancerGAT / TemporalEncoding / GraphAttentionLayer
- GDOBlock（新增参数 update_edges）
- EncoderModule / ProcessorModule / DecoderModule / PhysicsGuidedGNN
"""
import os
import math
from collections import deque

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, TransformerConv
from torch_geometric.utils import softmax

# =========================
# 新增：专用特征编码器
# =========================

class StaticFeatureEncoder(nn.Module):
    """编码静态特征 (地形, 土地利用等)"""
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, output_dim * 2),
            nn.LayerNorm(output_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim * 2, output_dim),
        )

    def forward(self, x_static):
        return self.encoder(x_static)

class DynamicFeatureEncoder(nn.Module):
    """编码动态特征 (降雨, 温度等) 和时间编码"""
    def __init__(self, dynamic_dim, time_dim, output_dim, dropout=0.1):
        super().__init__()
        input_dim = dynamic_dim + time_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, output_dim * 2),
            nn.LayerNorm(output_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim * 2, output_dim),
        )

    def forward(self, x_dynamic, x_time):
        x_combined = torch.cat([x_dynamic, x_time], dim=-1)
        return self.encoder(x_combined)

class HistoryEncoder(nn.Module):
    """使用 LSTM 编码历史水位序列，提取其时序模式"""
    def __init__(self, input_dim, output_dim, num_layers=1, dropout=0.1):
        super().__init__()
        # input_dim 通常为1，因为每个时间步只有一个水位值
        # output_dim 是 LSTM 的隐藏状态维度
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=output_dim,
            num_layers=num_layers,
            batch_first=True, # 输入形状 (batch, seq_len, features)
            dropout=dropout if num_layers > 1 else 0
        )
        self.output_norm = nn.LayerNorm(output_dim)

    def forward(self, x_history):
        # x_history 的形状是 (num_nodes, history_len)
        # LSTM 需要 (batch, seq_len, input_dim)
        # 我们将 num_nodes 视为 batch size
        x_history = x_history.unsqueeze(-1) # -> (num_nodes, history_len, 1)

        # LSTM 返回 (output, (h_n, c_n))
        # 我们只需要最后一个时间步的隐藏状态 h_n
        _, (h_n, _) = self.lstm(x_history)

        # h_n 的形状是 (num_layers, num_nodes, hidden_size)
        # 我们取最后一层的隐藏状态
        last_layer_hidden_state = h_n[-1, :, :] # -> (num_nodes, hidden_size)

        return self.output_norm(last_layer_hidden_state)
# =========================
# 空间与时间编码模块（原样保留）
# =========================
import torch
import torch.nn as nn
import math

class SpatialEncoding(nn.Module):
    """
    基于随机傅里叶特征 (Random Fourier Features) 的空间编码器。
    这比标准的 Transformer PE 更适合 [0,1] 范围的连续坐标。
    """
    def __init__(self, d_model, coord_min, coord_max, sigma=10.0):
        super().__init__()
        self.d_model = d_model
        assert d_model % 2 == 0, "d_model must be even"
        
        # --- 【核心修复】 ---
        # 1. 先将 coord_min 转换成 Tensor 并注册
        coord_min_tensor = torch.tensor(coord_min, dtype=torch.float)
        self.register_buffer('coord_min', coord_min_tensor)

        # 2. 然后用它来计算 coord_range
        coord_range_tensor = torch.tensor(coord_max, dtype=torch.float) - self.coord_min
        
        # 3. 避免除以0，并注册 coord_range
        coord_range_tensor[coord_range_tensor == 0] = 1.0
        self.register_buffer('coord_range', coord_range_tensor)
        # --- 修复结束 ---

        # 随机傅里叶特征矩阵 B
        self.register_buffer('B', torch.randn((2, d_model // 2)) * sigma)

    def forward(self, coords):
        if coords.size(0) == 0:
            return torch.empty(0, self.d_model, device=coords.device)
            
        # 归一化到 [0, 1]
        coords_norm = (coords.float() - self.coord_min) / self.coord_range
        
        # 映射到 [-1, 1]
        coords_norm = 2 * coords_norm - 1

        # 投影
        projected = (2 * math.pi * coords_norm) @ self.B

        # 拼接 Sin 和 Cos
        encoding = torch.cat([torch.sin(projected), torch.cos(projected)], dim=-1)
        
        return encoding




class EdgeEnhancerGAT(nn.Module):
    """边增强：先用 GAT 更新节点，再用 [src, dst, e] 通过 MLP 增强边。
    保留为一次性前置增强（不加门控）。
    """
    def __init__(self, node_dim, edge_dim, heads=4, dropout=0.1):
        super().__init__()
        self.gat = GATConv(
            in_channels=node_dim,
            out_channels=node_dim // heads,
            heads=heads,
            dropout=dropout,
            concat=True,
        )
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * node_dim + edge_dim, edge_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(edge_dim * 2, edge_dim),
        )

    def forward(self, x, edge_index, edge_attr):
        x_updated = self.gat(x, edge_index)
        row, col = edge_index
        src_feat = x_updated[row]
        dst_feat = x_updated[col]
        edge_input = torch.cat([src_feat, dst_feat, edge_attr], dim=-1)
        edge_attr_updated = self.edge_mlp(edge_input)
        return edge_attr_updated


class TemporalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, time_steps):
        return self.pe[time_steps]


class GraphAttentionLayer(nn.Module):
    def __init__(self, node_dim, nheads=4, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(...)
        self.norm = nn.LayerNorm(node_dim) # 保持不变
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, coords):
        # 1. 先 Norm
        x_norm = self.norm(x) 
        
        # 2. 计算 Attention (使用归一化后的 x)
        if x.dim() == 2:
            x_batch = x_norm.unsqueeze(0)
        else:
            x_batch = x_norm
            
        attn_out, _ = self.attention(x_batch, x_batch, x_batch)
        
        # 3. 残差连接 (加到原始 x 上)
        if x.dim() == 2:
            x = x + self.dropout(attn_out.squeeze(0))
        else:
            x = x + self.dropout(attn_out)
            
        return x # 输出不再做 Norm，留给下一个 Block 的开头


# =========================
# GDOBlock（加门控 + 可开关边更新）
# =========================
# 修改：GDOBlock (增加动态边权重)
# =========================
class GDOBlock(nn.Module):
    """Graph Deep Operator (核心图块) - 增加动态边权重模块"""
    def __init__(self, node_dim, edge_dim, nheads=4, dropout=0.1, update_edges=False):
        super().__init__()
        self.node_dim = node_dim
        self.edge_dim = edge_dim
        self.nheads = nheads
        self.update_edges = update_edges

        self.norm_nodes_in = nn.LayerNorm(node_dim)
        self.norm_edges_in = nn.LayerNorm(edge_dim)

        # 动态边权重模块 (新)
        # 它会根据相连节点的当前状态，生成一个调制因子
        self.dynamic_edge_mlp = nn.Sequential(
            nn.Linear(node_dim * 2, edge_dim),
            nn.GELU(),
            nn.Linear(edge_dim, 1),
            nn.Sigmoid() # 输出一个 0-1 之间的权重
        )

        assert node_dim % nheads == 0
        self.graph_operator = TransformerConv(
            in_channels=node_dim,
            out_channels=node_dim // nheads,
            heads=nheads,
            dropout=dropout,
            edge_dim=edge_dim,
            concat=True,
            beta=True,
        )
        self.dropout = nn.Dropout(dropout)

        if update_edges:
            self.edge_update_mlp = nn.Sequential(
                nn.Linear(node_dim * 2 + edge_dim, edge_dim * 2), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(edge_dim * 2, edge_dim)
            )
            self.edge_gate = nn.Parameter(torch.tensor(-1.0))

        self.norm_edges_out = nn.LayerNorm(edge_dim)

        self.ffn = nn.Sequential(
            nn.LayerNorm(node_dim), nn.Linear(node_dim, node_dim * 4),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(node_dim * 4, node_dim)
        )
        
        #self.norm_nodes_out = nn.LayerNorm(node_dim)

    def forward(self, x, edge_index, edge_attr):
        N, E = x.size(0), edge_index.size(1)
        if N == 0 or E == 0:
            return x, edge_attr

        row, col = edge_index
        x_norm = self.norm_nodes_in(x)
        e_norm = self.norm_edges_in(edge_attr)

        # --- 动态边权重计算 (新) ---
        src_for_edge, dst_for_edge = x_norm[row], x_norm[col]
        edge_runtime_features = torch.cat([src_for_edge, dst_for_edge], dim=-1)
        dynamic_weights = self.dynamic_edge_mlp(edge_runtime_features) # -> (E, 1)

        # 调制静态边属性
        e_for_conv = e_norm * dynamic_weights

        # --- 节点更新 ---
        x_res = x
        x_gdo = self.graph_operator(x_norm, edge_index, edge_attr=e_for_conv)
        x = x_res + self.dropout(x_gdo)
        x = x + self.dropout(self.ffn(x))
        
        #x = self.norm_nodes_out(x)

        # --- 边更新 (可选) ---
        e_res = edge_attr
        if self.update_edges:
            src, dst = x[row], x[col]
            m_in = torch.cat([src, dst, e_norm], dim=-1)
            delta_e = self.edge_update_mlp(m_in)
            gate = torch.sigmoid(self.edge_gate)
            edge_attr = e_res + gate * self.dropout(delta_e)

        edge_attr = self.norm_edges_out(edge_attr)
        return x, edge_attr

# =========================
# 编码器：两层 GDO（第一层更新边，第二层不更新）
# =========================
class EncoderModule(nn.Module):
    """编码器：处理原始观测数据，估计初始状态"""
    def __init__(self, input_dim, hidden_dim, edge_dim, coord_min, coord_max, dropout=0.1):
        super().__init__()
        self.spatial_encoder = SpatialEncoding(hidden_dim, coord_min, coord_max)
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        # 方案 B：只让第一层更新边，第二层不更新
        self.gdo_layers = nn.ModuleList([
            GDOBlock(hidden_dim, edge_dim, nheads=4, dropout=dropout, update_edges=True),
            GDOBlock(hidden_dim, edge_dim, nheads=4, dropout=dropout, update_edges=False),
        ])
        self.spatial_attention = GraphAttentionLayer(hidden_dim, nheads=4, dropout=dropout)
        self.edge_enhancer = EdgeEnhancerGAT(hidden_dim, edge_dim, heads=4, dropout=dropout)
        self.output_norm = nn.LayerNorm(hidden_dim)

    def forward(self, data):
        x, edge_index, edge_attr, coords = data.x, data.edge_index, data.edge_attr, data.coords
        x = self.input_proj(x)
        x = x + self.spatial_encoder(coords)
        # 前置一次边增强（无门控）
        edge_attr = self.edge_enhancer(x, edge_index, edge_attr)
        for layer in self.gdo_layers:
            x, edge_attr = layer(x, edge_index, edge_attr)
        #x = self.spatial_attention(x, coords)
        x = self.output_norm(x)
        return x, edge_attr


# =========================
# Processor：LSTM + 3 层 GDO（前两层不更新，最后一层更新）
# =========================
class ProcessorModule(nn.Module):
    def __init__(self, hidden_dim, edge_dim, num_gdo_layers=3, nheads=4, dropout=0.1, lstm_layers=1):
        super().__init__()
        assert num_gdo_layers >= 1, "num_gdo_layers 必须 >= 1"
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0,
        )
        self.h_state = None
        self.c_state = None
        self.dropout = nn.Dropout(dropout)
        self.norm_lstm_out = nn.LayerNorm(hidden_dim)

        # 方案 B：
        layers = []
        for i in range(num_gdo_layers):
            layers.append(GDOBlock(hidden_dim, edge_dim, nheads=nheads, dropout=dropout))
        self.gdo_layers = nn.ModuleList(layers)
        self.norm_final = nn.LayerNorm(hidden_dim)

    def reset_states(self):
        self.h_state = None
        self.c_state = None

    def forward(self, x, edge_index, edge_attr, use_stored_state=False):
        x_seq = x.unsqueeze(1)
        if use_stored_state and self.h_state is not None and self.c_state is not None:
            lstm_out, (new_h, new_c) = self.lstm(x_seq, (self.h_state, self.c_state))
            self.h_state, self.c_state = new_h, new_c
        else:
            lstm_out, (new_h, new_c) = self.lstm(x_seq)
            if use_stored_state:
                self.h_state, self.c_state = new_h, new_c
        x_temporal = lstm_out.squeeze(1)
        x = x + self.dropout(x_temporal)
        x = self.norm_lstm_out(x)
        for layer in self.gdo_layers:
            x, edge_attr = layer(x, edge_index, edge_attr)
        x = self.norm_final(x)
        return x, edge_attr


# =========================
# 解码器（原样保留）
# =========================
class NHiTSBlock(nn.Module):
    """N-HiTS 的一个基本块，用于生成部分预测。"""
    def __init__(self, input_dim, forecast_len, basis_degree, num_layers, dropout):
        super().__init__()
        self.forecast_len = forecast_len
        self.basis_degree = basis_degree

        # MLP层来处理输入特征
        layers = [nn.Linear(input_dim, input_dim), nn.GELU(), nn.Dropout(dropout)]
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(input_dim, input_dim), nn.GELU(), nn.Dropout(dropout)])
        self.mlp = nn.Sequential(*layers)

        # 预测头，用于生成基函数的系数
        self.forecast_head = nn.Linear(input_dim, basis_degree + 1)

        # 基函数层
        self.basis_layer = PolynomialBasisLayer(basis_degree, forecast_len)

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): 输入特征，形状为 [num_nodes, input_dim]。
        Returns:
            torch.Tensor: 此块的预测序列，形状为 [num_nodes, forecast_len]。
        """
        # 通过 MLP 处理输入
        x = self.mlp(x)

        # 计算预测系数
        theta_forecast = self.forecast_head(x)

        # 生成预测
        forecast = self.basis_layer(theta_forecast)

        return forecast
class MultiScaleNHiTSDecoder(nn.Module):
    """
    分层N-HiTS解码器，专为地下水位预测设计
    - Stack 1: 长期趋势 (低频)
    - Stack 2: 季节性模式 (中频)
    - Stack 3: 短期波动 (高频)
    """
    def __init__(
        self,
        hidden_dim=128,
        forecast_len=12,
        dropout=0.1
    ):
        super().__init__()
        self.forecast_len = forecast_len

        # ========== Stack 1: 长期趋势 (年际变化) ==========
        # 使用低阶多项式 + 大感受野
        self.trend_stack = nn.ModuleList([
            NHiTSBlock(
                input_dim=hidden_dim,
                forecast_len=forecast_len,
                basis_degree=2,  # 低阶多项式捕捉平滑趋势
                num_layers=3,
                dropout=dropout
            ) for _ in range(2)  # 2个block
        ])

        # ========== Stack 2: 季节性模式 (月度周期) ==========
        # 使用中阶多项式 + 傅里叶基
        self.seasonal_stack = nn.ModuleList([
            SeasonalNHiTSBlock(
                input_dim=hidden_dim,
                forecast_len=forecast_len,
                basis_degree=4,
                num_fourier_terms=3,  # 捕捉周期性
                num_layers=2,
                dropout=dropout
            ) for _ in range(3)  # 3个block
        ])

        # ========== Stack 3: 短期波动 (突变响应) ==========
        # 使用高阶多项式 + 注意力机制
        self.shortterm_stack = nn.ModuleList([
            ShortTermNHiTSBlock(
                input_dim=hidden_dim,
                forecast_len=forecast_len,
                basis_degree=6,  # 高阶捕捉快速变化
                num_layers=2,
                dropout=dropout
            ) for _ in range(4)  # 4个block
        ])

        # ========== 物理约束增强模块 ==========
        self.physics_modulator = PhysicsModulator(hidden_dim, forecast_len)

        # ========== 自适应权重融合 ==========
        self.stack_fusion = nn.Sequential(
            nn.Linear(3, 16),
            nn.GELU(),
            nn.Linear(16, 3),
            nn.Softmax(dim=-1)
        )

    def forward(self, encoder_state, gradient_gt=None, curvature_gt=None):
        """
        Args:
            encoder_state: [num_nodes, hidden_dim] - GNN编码的特征
            gradient_gt: [num_nodes] - 地形梯度(训练时可用)
            curvature_gt: [num_nodes] - 地形曲率(训练时可用)
        """
        num_nodes = encoder_state.size(0)
        device = encoder_state.device

        # ========== 1. 趋势预测 ==========
        trend_residual = encoder_state
        trend_forecast = torch.zeros(num_nodes, self.forecast_len, device=device)

        for block in self.trend_stack:
            block_pred = block(trend_residual)
            trend_forecast += block_pred
            # 更新残差(简化版本)
            trend_residual = trend_residual - 0.1 * torch.mean(
                block_pred.unsqueeze(-1) * encoder_state.unsqueeze(1),
                dim=1
            )

        # ========== 2. 季节性预测 ==========
        seasonal_residual = encoder_state
        seasonal_forecast = torch.zeros(num_nodes, self.forecast_len, device=device)

        for block in self.seasonal_stack:
            block_pred = block(seasonal_residual)
            seasonal_forecast += block_pred
            seasonal_residual = seasonal_residual - 0.05 * torch.mean(
                block_pred.unsqueeze(-1) * encoder_state.unsqueeze(1),
                dim=1
            )

        # ========== 3. 短期波动预测 ==========
        shortterm_residual = encoder_state
        shortterm_forecast = torch.zeros(num_nodes, self.forecast_len, device=device)

        for block in self.shortterm_stack:
            block_pred = block(shortterm_residual)
            shortterm_forecast += block_pred
            shortterm_residual = shortterm_residual - 0.02 * torch.mean(
                block_pred.unsqueeze(-1) * encoder_state.unsqueeze(1),
                dim=1
            )

        # ========== 4. 物理约束调制 ==========
        if gradient_gt is not None and curvature_gt is not None:
            physics_correction = self.physics_modulator(
                encoder_state, gradient_gt, curvature_gt
            )
        else:
            physics_correction = 0.0

        # ========== 5. 自适应融合 ==========
        # 计算每个stack的贡献权重
        stack_features = torch.stack([
            torch.mean(torch.abs(trend_forecast), dim=-1),
            torch.mean(torch.abs(seasonal_forecast), dim=-1),
            torch.mean(torch.abs(shortterm_forecast), dim=-1)
        ], dim=-1)  # [num_nodes, 3]

        fusion_weights = self.stack_fusion(stack_features)  # [num_nodes, 3]

        # 加权融合
        final_forecast = (
            fusion_weights[:, 0:1] * trend_forecast +
            fusion_weights[:, 1:2] * seasonal_forecast +
            fusion_weights[:, 2:3] * shortterm_forecast +
            physics_correction
        )

        return {
            'output': final_forecast,
            'trend': trend_forecast,
            'seasonal': seasonal_forecast,
            'shortterm': shortterm_forecast,
            'fusion_weights': fusion_weights
        }


class SeasonalNHiTSBlock(nn.Module):
    """带傅里叶基的季节性预测块"""
    def __init__(self, input_dim, forecast_len, basis_degree, num_fourier_terms, num_layers, dropout):
        super().__init__()
        self.forecast_len = forecast_len

        # MLP处理
        self.mlp = nn.Sequential(*[
            layer for _ in range(num_layers)
            for layer in [nn.Linear(input_dim, input_dim), nn.GELU(), nn.Dropout(dropout)]
        ])

        # 多项式基
        self.poly_head = nn.Linear(input_dim, basis_degree + 1)
        self.poly_basis = PolynomialBasisLayer(basis_degree, forecast_len)

        # 傅里叶基
        self.fourier_head = nn.Linear(input_dim, num_fourier_terms * 2)
        self.register_buffer('fourier_basis', self._create_fourier_basis(forecast_len, num_fourier_terms))

    def _create_fourier_basis(self, length, num_terms):
        """创建傅里叶基函数"""
        t = torch.linspace(0, 2 * 3.14159, length)
        basis = []
        for k in range(1, num_terms + 1):
            basis.append(torch.sin(k * t))
            basis.append(torch.cos(k * t))
        return torch.stack(basis, dim=0)  # [2*num_terms, length]

    def forward(self, x):
        x = self.mlp(x)

        # 多项式分量
        poly_coef = self.poly_head(x)
        poly_pred = self.poly_basis(poly_coef)

        # 傅里叶分量
        fourier_coef = self.fourier_head(x)  # [num_nodes, 2*num_terms]
        fourier_pred = torch.matmul(fourier_coef, self.fourier_basis)  # [num_nodes, length]

        return poly_pred + fourier_pred

class PolynomialBasisLayer(nn.Module):
    """生成多项式基函数。"""
    def __init__(self, degree, forecast_len):
        super().__init__()
        # 创建时间步向量 [0, 1, 2, ..., forecast_len-1]
        t = torch.linspace(0, 1, forecast_len, dtype=torch.float32)

        # 使用广播机制计算多项式基 [t^0, t^1, ..., t^degree]
        # powers shape: [degree + 1]
        powers = torch.arange(degree + 1, dtype=torch.float32)
        # t.unsqueeze(-1) shape: [forecast_len, 1]
        # basis shape: [forecast_len, degree + 1]
        basis = t.unsqueeze(-1) ** powers

        # 注册为 buffer，这样它会被移动到 GPU，但不是模型参数
        self.register_buffer('basis', basis)

    def forward(self, theta):
        """
        Args:
            theta (torch.Tensor): 系数，形状为 [num_nodes, degree + 1]。
        Returns:
            torch.Tensor: 预测序列，形状为 [num_nodes, forecast_len]。
        """
        # 矩阵乘法: (num_nodes, degree + 1) @ (degree + 1, forecast_len)
        return torch.matmul(theta, self.basis.T)


class ShortTermNHiTSBlock(nn.Module):
    """带注意力的短期预测块"""
    def __init__(self, input_dim, forecast_len, basis_degree, num_layers, dropout):
        super().__init__()

        self.mlp = nn.Sequential(*[
            layer for _ in range(num_layers)
            for layer in [nn.Linear(input_dim, input_dim), nn.GELU(), nn.Dropout(dropout)]
        ])

        # 时间步特定的注意力
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=input_dim,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )

        self.forecast_head = nn.Linear(input_dim, basis_degree + 1)
        self.basis_layer = PolynomialBasisLayer(basis_degree, forecast_len)

    def forward(self, x):
        # MLP处理
        x_processed = self.mlp(x)

        # 自注意力(模拟时间依赖)
        x_seq = x_processed.unsqueeze(1)  # [num_nodes, 1, hidden_dim]
        attn_out, _ = self.temporal_attention(x_seq, x_seq, x_seq)
        x_enhanced = x_processed + attn_out.squeeze(1)

        # 预测
        theta = self.forecast_head(x_enhanced)
        return self.basis_layer(theta)


class PhysicsModulator(nn.Module):
    """物理约束调制器"""
    def __init__(self, hidden_dim, forecast_len):
        super().__init__()

        # 梯度影响网络
        self.gradient_net = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, forecast_len)
        )

        # 曲率影响网络
        self.curvature_net = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, forecast_len)
        )

        # 融合门控
        self.fusion_gate = nn.Sequential(
            nn.Linear(forecast_len * 2, forecast_len),
            nn.Sigmoid()
        )

    def forward(self, x, gradient, curvature):
        """
        利用地形物理特性调制预测
        梯度大 -> 水位变化快
        曲率大 -> 汇聚/发散效应强
        """
        # 拼接特征
        x_grad = torch.cat([x, gradient.unsqueeze(-1)], dim=-1)
        x_curv = torch.cat([x, curvature.unsqueeze(-1)], dim=-1)

        # 计算调制
        grad_effect = self.gradient_net(x_grad)
        curv_effect = self.curvature_net(x_curv)

        # 门控融合
        combined = torch.cat([grad_effect, curv_effect], dim=-1)
        gate = self.fusion_gate(combined)

        return gate * (grad_effect + curv_effect)

class SpatialDecoderCell(nn.Module):
    """
    一个解码器单元，功能类似于GRU单元，但在更新隐藏状态时加入了图卷积。
    """
    def __init__(self, input_dim, hidden_dim, edge_dim, n_heads=4, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim

        # GNN层，用于空间信息传播
        self.gnn = GATv2Conv(
            in_channels=hidden_dim + input_dim, # 输入 = 当前隐藏状态 + 新输入
            out_channels=hidden_dim,
            heads=n_heads,
            concat=True,
            edge_dim=edge_dim,
            dropout=dropout
        )

        gnn_output_dim = hidden_dim * n_heads

        # GRU的门控机制: 重置门 (reset gate) 和 更新门 (update gate)
        self.reset_gate_mlp = nn.Linear(gnn_output_dim + hidden_dim, hidden_dim)
        self.update_gate_mlp = nn.Linear(gnn_output_dim + hidden_dim, hidden_dim)
        self.candidate_state_mlp = nn.Linear(gnn_output_dim + hidden_dim, hidden_dim)

    def forward(self, x_t, h_prev, edge_index, edge_attr):
        """
        Args:
            x_t (Tensor): 当前时间步的输入 [num_nodes, input_dim]
            h_prev (Tensor): 上一时间步的隐藏状态 [num_nodes, hidden_dim]
            edge_index, edge_attr: 图结构
        """
        # 1. 组合输入进行GNN计算
        gnn_input = torch.cat([h_prev, x_t], dim=-1)
        gnn_out = self.gnn(gnn_input, edge_index, edge_attr)

        # 2. 计算门控
        combined_for_gates = torch.cat([gnn_out, h_prev], dim=-1)

        update_gate = torch.sigmoid(self.update_gate_mlp(combined_for_gates))
        reset_gate = torch.sigmoid(self.reset_gate_mlp(combined_for_gates))

        # 3. 计算候选隐藏状态
        candidate_input = torch.cat([gnn_out, reset_gate * h_prev], dim=-1)
        candidate_h = torch.tanh(self.candidate_state_mlp(candidate_input))

        # 4. 更新隐藏状态
        h_next = (1 - update_gate) * h_prev + update_gate * candidate_h

        return h_next

# new_modules.py (或者直接添加到你的代码中)

class HistorySequenceEncoder(nn.Module):
    """
    独立编码每个节点的历史水位序列 (x_history_encoder)，
    生成一个固定维度的历史水位摘要向量。
    """
    def __init__(self, history_input_dim, hidden_dim, num_layers=1, dropout=0.1, mode='LSTM'):
        super().__init__()
        self.mode = mode.upper()

        # 历史水位通常是1维的 (水位值)
        # self.input_proj = nn.Linear(history_input_dim, hidden_dim) # 如果需要先投影

        if self.mode == 'LSTM':
            self.rnn = nn.LSTM(
                input_size=history_input_dim, # 直接使用原始维度
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0
            )
        elif self.mode == 'GRU':
            self.rnn = nn.GRU(
                input_size=history_input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0
            )
        else:
            raise ValueError(f"HistorySequenceEncoder不支持的模式: {mode}. 请使用 'LSTM' 或 'GRU'.")

        self.output_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x_history_encoder_seq):
        """
        Args:
            x_history_encoder_seq (torch.Tensor): 形状为 [num_nodes, enc_len, history_input_dim]
                                                  其中 history_input_dim 通常为 1。
        Returns:
            torch.Tensor: 形状为 [num_nodes, hidden_dim]，表示历史水位的摘要。
        """
        # x = self.input_proj(x_history_encoder_seq) # 如果需要投影

        if self.mode == 'LSTM':
            _, (h_n, _) = self.rnn(x_history_encoder_seq)
        else: # GRU
            _, h_n = self.rnn(x_history_encoder_seq)

        # 提取最后一层RNN的最终隐藏状态
        history_summary = h_n[-1, :, :] # [num_nodes, hidden_dim]

        return self.output_norm(history_summary)

# physics_guided_gnn.py
import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv
class DynamicEdgeUpdater(nn.Module):
    """
    根据未来的节点动态特征（如抽水、灌溉计划）更新边属性。
    这解决了Decoder使用静态边的限制。
    """
    def __init__(self, edge_dim, node_dynamic_dim, dropout=0.1):
        super().__init__()
        # 输入: 原边特征 + 源节点未来特征 + 目标节点未来特征
        input_dim = edge_dim + 2 * node_dynamic_dim
        
        self.update_mlp = nn.Sequential(
            nn.Linear(input_dim, edge_dim * 2),
            nn.LayerNorm(edge_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(edge_dim * 2, edge_dim)
        )
        # 门控机制，控制更新幅度
        self.gate = nn.Sequential(
            nn.Linear(input_dim, edge_dim),
            nn.Sigmoid()
        )

    def forward(self, edge_attr, edge_index, x_dynamic_future_t):
        """
        Args:
            edge_attr: 上一步的边特征 [num_edges, edge_dim]
            edge_index: 图连接 [2, num_edges]
            x_dynamic_future_t: 当前预测步的节点特征 [num_nodes, dyn_dim]
                                (包含降雨、温度、未来的抽水/灌溉等)
        """
        row, col = edge_index
        
        # 获取边的源节点和目标节点的未来动态特征
        src_feat = x_dynamic_future_t[row]
        dst_feat = x_dynamic_future_t[col]
        
        # 拼接信息
        concat_feat = torch.cat([edge_attr, src_feat, dst_feat], dim=-1)
        
        # 计算更新量
        delta_edge = self.update_mlp(concat_feat)
        update_gate = self.gate(concat_feat)
        
        # 残差更新：保留部分旧的物理结构信息，融入新的动态影响
        new_edge_attr = edge_attr + update_gate * delta_edge
        
        return new_edge_attr

class GraphProcessorDecoder(nn.Module):
    """
    升级版图解码器：
    1. 使用与Encoder相同的 GDOBlock (TransformerConv) 替代 GATv2。
    2. 引入动态边更新，使未来的抽水计划能改变图结构。
    3. 依然使用 GRUCell 保持时序记忆。
    """
    def __init__(self, hidden_dim, edge_dim, forecast_len, dynamic_feat_dim, time_feat_dim, 
                 num_gdo_layers=2, nheads=4, dropout=0.1):
        super().__init__()
        self.forecast_len = forecast_len
        self.hidden_dim = hidden_dim
        
        # 1. 动态边更新器
        # 输入特征维度 = dynamic_feat_dim (降雨/抽水等) + time_feat_dim (时间编码)
        future_feat_dim = dynamic_feat_dim + time_feat_dim
        self.edge_updater = DynamicEdgeUpdater(edge_dim, future_feat_dim, dropout)
        
        # 2. 输入融合层
        self.input_proj = nn.Sequential(
            nn.Linear(hidden_dim + future_feat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        # 3. GDOBlock 堆叠 (与 Encoder 结构对齐)
        # 这里使用 ModuleList 来手动控制每一层的边更新逻辑
        self.gdo_layers = nn.ModuleList()
        for i in range(num_gdo_layers):
            # 只有最后一层更新边，或者根据需求配置
            # 这里我们设为 False，因为我们在 GDO 外部显式使用了 edge_updater
            self.gdo_layers.append(
                GDOBlock(hidden_dim, edge_dim, nheads=nheads, dropout=dropout, update_edges=False)
            )
            
        # 4. 时序记忆单元
        self.gru_cell = nn.GRUCell(hidden_dim, hidden_dim)
        
        # 5. 输出头
        self.output_head = nn.Linear(hidden_dim, 1)

    def forward(self, h_state, edge_index, edge_attr, x_dynamic_future, x_time_future):
        """
        h_state: Encoder 最终状态
        edge_attr: Encoder 最终输出的边特征 (作为起点)
        """
        predictions = []
        curr_h = h_state
        curr_edge_attr = edge_attr # 初始化边特征
        
        for t in range(self.forecast_len):
            # --- A. 准备未来特征 ---
            # 包含未来的抽水、降雨计划和时间信息
            future_feats_t = torch.cat([
                x_dynamic_future[:, t, :],
                x_time_future[:, t, :]
            ], dim=-1)
            
            # --- B. 动态更新边属性 (关键修改) ---
            # 根据 T 时刻的抽水计划，更新 T 时刻的水文连通性
            curr_edge_attr = self.edge_updater(curr_edge_attr, edge_index, future_feats_t)
            
            # --- C. 节点特征融合 ---
            # 将上一时刻的隐状态与未来的强迫数据融合
            combined_input = torch.cat([curr_h, future_feats_t], dim=-1)
            x = self.input_proj(combined_input)
            
            # --- D. GDOBlock 空间处理 (与 Encoder 操作相同) ---
            # 使用 TransformerConv 进行深层空间聚合
            for gdo in self.gdo_layers:
                # 注意：这里我们传入更新后的 curr_edge_attr
                x, _ = gdo(x, edge_index, curr_edge_attr)
            
            # --- E. GRU 时序演化 ---
            curr_h = self.gru_cell(x, curr_h)
            
            # --- F. 输出 ---
            pred_t = self.output_head(curr_h)
            predictions.append(pred_t)
            
        return torch.cat(predictions, dim=-1)

class PhysicsGuidedGNN(nn.Module):
    """
    融合了 GNN 和 分层N-HiTS 的时空预测模型。
    增强了对节点自身时间序列模式的捕捉能力，尤其是历史水位。
    """
    def __init__(
        self,
        static_feat_dim,
        dynamic_feat_dim,
        time_feat_dim,
        # 新增一个参数用于 HistorySequenceEncoder
        history_input_dim=1, # 历史水位通常是1维
        hidden_dim=128,
        edge_dim=7,
        coord_min=None,
        coord_max=None,
        dropout=0.1,
        forecast_len=12,
        num_rnn_layers_history=1, # 用于HistorySequenceEncoder的RNN层数
        rnn_mode_history='LSTM',  # HistorySequenceEncoder的RNN类型
    ):
        super().__init__()
        os.environ["PYTORCH_ENABLE_MEM_EFF_ATTENTION"] = "0"
        os.environ["PYTORCH_DISABLE_FLASH_ATTENTION"] = "1"
        self.hidden_dim = hidden_dim
        self.forecast_len = forecast_len
        # 【新增】：实例化物理适配层
        self.physics_adapter = PhysicsAdaptationLayer(initial_k_factor=0.5, initial_grad_factor=0.5)
        
        # 如果 PhysicsAdaptationLayer 输出 8 维特征，确保这里的 edge_dim 匹配
        # 或者在 adapter 后加一个 Linear 投影层回到原来的 edge_dim
        self.edge_proj = nn.Linear(7, edge_dim) # 假设 adapter 输出 8 维

        # --- GNN Encoder 部分 ---
        self.static_encoder = StaticFeatureEncoder(static_feat_dim, hidden_dim, dropout)
        self.dynamic_encoder = DynamicFeatureEncoder(dynamic_feat_dim, time_feat_dim, hidden_dim, dropout)
        # [[修改点 1]]: 新增 HistorySequenceEncoder 模块
        self.history_sequence_encoder = HistorySequenceEncoder(
            history_input_dim=history_input_dim,
            hidden_dim=hidden_dim, # 历史摘要的维度与GNN hidden_dim 相同
            num_layers=num_rnn_layers_history,
            dropout=dropout,
            mode=rnn_mode_history
        )
        self.spatial_encoder = SpatialEncoding(hidden_dim, coord_min, coord_max)
        self.history_step_proj = nn.Linear(1, hidden_dim) # 新增：单步投影

        # [[修改点 2]]: 调整融合层的输入维度
        # 现在 fusion_layer 接收：静态Embed + 动态Embed + 历史序列摘要 + 空间Embed
        # 即 hidden_dim + hidden_dim + hidden_dim + hidden_dim = hidden_dim * 4
        # 如果 HistorySequenceEncoder 的 output_dim 不同，需要对应调整
        self.fusion_layer = nn.Sequential(
            nn.Linear(hidden_dim * 5, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        self.encoder_processor = ProcessorModule(hidden_dim, edge_dim, num_gdo_layers=3, dropout=dropout)

        # 1. N-HiTS Decoder (Path A: 时频分解)
        self.nhits_decoder = MultiScaleNHiTSDecoder(
            hidden_dim=hidden_dim,
            forecast_len=forecast_len,
            dropout=dropout
        )
        
        # 2. GNN Recurrent Decoder (Path B: 空间修正) - 新增!
        self.spatial_decoder = GraphProcessorDecoder(
            hidden_dim=hidden_dim,
            edge_dim=edge_dim, 
            forecast_len=forecast_len,
            dynamic_feat_dim=dynamic_feat_dim,
            time_feat_dim=time_feat_dim,
            num_gdo_layers=2, # 使用2层GDO，保持计算效率与深度的平衡
            dropout=dropout
        )
        
        # 3. 融合门控 (可选，也可以直接相加)
        # 用一个可学习参数控制两个分支的权重，初始偏向 N-HiTS (权重设为 0.1 对 GNN)
        self.spatial_weight = nn.Parameter(torch.tensor(0.1))

        # 辅助任务头 (改为预测序列)
        self.gradient_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, forecast_len)
        )

        self.curvature_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, forecast_len)
        )

        # [[修改点 3]]: 新增解码器阶段的特征融合层，以更直接地利用静态和未来的动态特征
        # 这个层将结合每个时间步的 GNN 节点状态、静态特征、未来的动态特征和时间编码
        self.decoder_feature_fusion = nn.Sequential(
            nn.Linear(hidden_dim + static_feat_dim + dynamic_feat_dim + time_feat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim) # 再次投影到 hidden_dim
        )

    def forward(self, data):
        """
        模型的前向传播，包含分层残差解码。
        """
        edge_index = data.edge_index
        edge_attr_raw = data.edge_attr # 这是 DataProcessor 输出的原始“原材料”

        # 【修改点】：首先通过物理适配层，进行端到端参数计算
        edge_attr_modulated = self.physics_adapter(edge_attr_raw)
        
        # 投影到模型需要的维度 (比如 hidden_dim 或原本设定的 edge_dim)
        edge_attr = self.edge_proj(edge_attr_modulated)
        encoder_len = data.x_dynamic_encoder.size(1)
        num_nodes = data.x_static.size(0)

        # ========================= ENCODER =========================
        self.encoder_processor.reset_states()

        # [[修改点 4]]: 独立编码历史水位序列
        # data.x_history_encoder 形状是 [num_nodes, enc_len, 1]
        
        history_summary_embed = self.history_sequence_encoder(data.x_history_encoder) # [num_nodes, hidden_dim]

        static_embed = self.static_encoder(data.x_static) # [num_nodes, hidden_dim]
        spatial_embed = self.spatial_encoder(data.coords) # [num_nodes, hidden_dim]

        # 初始化节点状态
        current_node_state = None

        for t in range(encoder_len):
            dynamic_embed_t = self.dynamic_encoder(
                data.x_dynamic_encoder[:, t, :],
                data.x_time_encoder[:, t, :]
            ) # [num_nodes, hidden_dim]

            # 2. 【新增】获取当前时间步具体的历史水位值
            # data.x_history_encoder 形状: [nodes, seq_len, 1]
            history_val_t = data.x_history_encoder[:, t, :] 
            history_step_embed = self.history_step_proj(history_val_t)

            # 3. 组合特征 (加入 history_step_embed)
            combined_embed_t = torch.cat([
                static_embed,
                dynamic_embed_t,
                history_summary_embed, # 全局上下文
                history_step_embed,    # 【关键】当前步的具体数值
                spatial_embed
            ], dim=-1)

            x_t = self.fusion_layer(combined_embed_t) # [num_nodes, hidden_dim]

            # 运行GNN Processor，更新节点状态
            current_node_state, edge_attr = self.encoder_processor(x_t, edge_index, edge_attr, use_stored_state=True)

        # 解码器现在接收的是 `current_node_state`，这是 Encoder 最后一个时间步的输出
        # 但我们还需要在解码器的每个时间步传入未来的动态特征。

        # ========================= DECODER =========================

        # ========================= DECODER =========================
        
        # 准备解码器输入状态
        final_encoder_state = current_node_state + history_summary_embed

        # --- Path A: N-HiTS Decoder (时间维) ---
        # 捕捉多尺度时间模式
        nhits_outputs = self.nhits_decoder(
            encoder_state=final_encoder_state, 
            gradient_gt=data.grad if hasattr(data, 'grad') else None,
            curvature_gt=data.curv if hasattr(data, 'curv') else None
        )
        nhits_pred = nhits_outputs['output']

        # --- Path B: GNN Decoder (空间维) ---
        # 利用更新后的边属性和未来特征进行空间修正
        spatial_pred = self.spatial_decoder(
            h_state=final_encoder_state,
            edge_index=edge_index,
            edge_attr=edge_attr,  # Encoder 传来的边
            x_dynamic_future=data.x_dynamic_decoder, # 未来的抽水/灌溉/降雨
            x_time_future=data.x_time_decoder
        )

        # --- Fusion ---
        # 结合两者的预测
        # 使用 sigmoid 确保权重为正，或者直接相加
        final_forecast = nhits_pred + self.spatial_weight * spatial_pred

        # 辅助任务
        gradient_pred = self.gradient_head(current_node_state)
        curvature_pred = self.curvature_head(current_node_state)

        return {
            'output': final_forecast,
            'nhits_component': nhits_pred,       # 可用于分析
            'spatial_component': spatial_pred,   # 可用于分析
            'gradient': gradient_pred,
            'curvature': curvature_pred,
            'trend': nhits_outputs['trend'],
            'seasonal': nhits_outputs['seasonal'],
            'shortterm': nhits_outputs['shortterm'],
            'fusion_weights': nhits_outputs['fusion_weights']
        }

    def predict(self, data):
        """在评估模式下进行序列预测。"""
        self.eval()
        with torch.no_grad():
            predictions = self.forward(data)
        return predictions





## Load quick-test assets

In [ ]:
required = [QUICK_DATA_FILE, MODEL_FILE]
missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required quick-test assets:\n  - " + "\n  - ".join(missing) +
        "\n\nFirst run 00_prepare_quick_test_from_saved_files.ipynb."
    )

try:
    quick_test_data = torch.load(
        QUICK_DATA_FILE, map_location="cpu", weights_only=False
    )
except TypeError:
    quick_test_data = torch.load(QUICK_DATA_FILE, map_location="cpu")

if not isinstance(quick_test_data, (list, tuple)) or len(quick_test_data) == 0:
    raise RuntimeError("quick_test_data.pt is empty.")

print(f"Loaded {len(quick_test_data)} quick-test graph samples.")

for i, g in enumerate(quick_test_data):
    print(
        f"Sample {i}: nodes={g.num_nodes}, "
        f"edges={g.edge_index.shape[1]}, "
        f"y={tuple(g.y.shape)}"
    )

if MANIFEST_FILE.exists():
    with open(MANIFEST_FILE, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print("\nManifest:")
    print(json.dumps(manifest, indent=2, ensure_ascii=False))

## Instantiate the trained architecture and load weights

In [ ]:
sample_data = quick_test_data[0]

static_feat_dim = int(sample_data.x_static.shape[1])
dynamic_feat_dim = int(sample_data.x_dynamic_encoder.shape[2])
time_feat_dim = int(sample_data.x_time_encoder.shape[2])
history_input_dim = int(sample_data.x_history_encoder.shape[2])
forecast_len = int(sample_data.y.shape[1])
edge_dim = int(sample_data.edge_attr.shape[1])

# Each saved full graph contains the study-domain node coordinates.
coords = sample_data.coords.detach().cpu()
coord_min = coords.min(dim=0).values.tolist()
coord_max = coords.max(dim=0).values.tolist()

model = PhysicsGuidedGNN(
    static_feat_dim=static_feat_dim,
    dynamic_feat_dim=dynamic_feat_dim,
    time_feat_dim=time_feat_dim,
    history_input_dim=history_input_dim,
    hidden_dim=128,
    edge_dim=edge_dim,
    coord_min=coord_min,
    coord_max=coord_max,
    dropout=0.15,
    forecast_len=forecast_len,
).to(DEVICE)

state = torch.load(MODEL_FILE, map_location=DEVICE)

if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
    state = state["state_dict"]

if isinstance(state, dict) and any(k.startswith("module.") for k in state):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

model.load_state_dict(state, strict=True)
model.eval()

print("Model checkpoint loaded successfully.")
print("Forecast horizon:", forecast_len)

## Run inference

In [ ]:
pred_parts = []
true_parts = []

with torch.inference_mode():
    for i, graph_cpu in enumerate(quick_test_data):
        graph = graph_cpu.to(DEVICE)
        output = model.predict(graph)["output"]
        target = graph.y

        if output.shape != target.shape:
            raise RuntimeError(
                f"Shape mismatch at sample {i}: "
                f"prediction={tuple(output.shape)}, target={tuple(target.shape)}"
            )

        pred_parts.append(output.detach().cpu().numpy())
        true_parts.append(target.detach().cpu().numpy())

pred_norm = np.concatenate(pred_parts, axis=0)
true_norm = np.concatenate(true_parts, axis=0)

print("Inference completed.")
print("Prediction shape:", pred_norm.shape)
print("Target shape:", true_norm.shape)

## Compute reproducibility metrics

In [ ]:
def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if np.allclose(np.std(y_true), 0.0):
        return np.nan
    return r2_score(y_true, y_pred)

def nse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(denom, 0.0):
        return np.nan
    return 1.0 - np.sum((y_pred - y_true) ** 2) / denom

metrics = {
    "RMSE_normalized": float(
        np.sqrt(mean_squared_error(true_norm.ravel(), pred_norm.ravel()))
    ),
    "MAE_normalized": float(
        mean_absolute_error(true_norm.ravel(), pred_norm.ravel())
    ),
    "R2": float(safe_r2(true_norm, pred_norm)),
    "NSE": float(nse(true_norm, pred_norm)),
    "n_graph_samples": int(len(quick_test_data)),
    "n_node_forecast_values": int(true_norm.size),
    "forecast_horizon": int(true_norm.shape[1]),
}

physical_units_available = SCALER_FILE.exists()

if physical_units_available:
    scaler = joblib.load(SCALER_FILE)

    pred_real = scaler.inverse_transform(
        pred_norm.reshape(-1, 1)
    ).reshape(pred_norm.shape)

    true_real = scaler.inverse_transform(
        true_norm.reshape(-1, 1)
    ).reshape(true_norm.shape)

    metrics["RMSE_groundwater_unit"] = float(
        np.sqrt(mean_squared_error(true_real.ravel(), pred_real.ravel()))
    )
    metrics["MAE_groundwater_unit"] = float(
        mean_absolute_error(true_real.ravel(), pred_real.ravel())
    )

print("=== Quick-test metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.6f}")
    else:
        print(f"{k}: {v}")

if not physical_units_available:
    print("\nNote:")
    print("target_scaler.joblib was not available.")
    print("RMSE/MAE are therefore reported in normalized units.")
    print("R² and NSE remain valid because they are invariant to affine target scaling.")

## Metrics by forecast horizon

In [ ]:
rows = []

for h in range(true_norm.shape[1]):
    yt = true_norm[:, h]
    yp = pred_norm[:, h]

    row = {
        "forecast_step": h + 1,
        "RMSE_normalized": np.sqrt(mean_squared_error(yt, yp)),
        "MAE_normalized": mean_absolute_error(yt, yp),
        "R2": safe_r2(yt, yp),
        "NSE": nse(yt, yp),
    }

    if physical_units_available:
        yt_real = true_real[:, h]
        yp_real = pred_real[:, h]
        row["RMSE_groundwater_unit"] = np.sqrt(
            mean_squared_error(yt_real, yp_real)
        )
        row["MAE_groundwater_unit"] = mean_absolute_error(
            yt_real, yp_real
        )

    rows.append(row)

horizon_metrics = pd.DataFrame(rows)
display(horizon_metrics)

csv_path = ASSET_DIR / "quick_test_metrics_by_horizon.csv"
horizon_metrics.to_csv(csv_path, index=False)
print("Saved:", csv_path)

## Visual sanity check

In [ ]:
if physical_units_available:
    x = true_real.ravel()
    y = pred_real.ravel()
    xlab = "Observed groundwater head"
    ylab = "Predicted groundwater head"
else:
    x = true_norm.ravel()
    y = pred_norm.ravel()
    xlab = "Observed target (normalized)"
    ylab = "Predicted target (normalized)"

rng = np.random.default_rng(SEED)
max_points = 5000

if len(x) > max_points:
    idx = rng.choice(len(x), max_points, replace=False)
    x_plot = x[idx]
    y_plot = y[idx]
else:
    x_plot = x
    y_plot = y

plt.figure(figsize=(7, 7))
plt.scatter(x_plot, y_plot, s=8, alpha=0.35)

lo = float(min(np.min(x_plot), np.min(y_plot)))
hi = float(max(np.max(x_plot), np.max(y_plot)))
plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.5)

plt.xlabel(xlab)
plt.ylabel(ylab)
plt.title(f"Quick test: prediction vs observation\nR²={metrics['R2']:.4f}")
plt.tight_layout()

plot_path = ASSET_DIR / "quick_test_prediction_scatter.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot_path)

## Save / compare reference metrics

In [ ]:
# Author: set True once after finalizing the released checkpoint/assets.
# Before submission, return it to False.
SAVE_REFERENCE = False

if SAVE_REFERENCE:
    with open(REFERENCE_FILE, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print("Saved reference metrics:", REFERENCE_FILE)

elif REFERENCE_FILE.exists():
    with open(REFERENCE_FILE, "r", encoding="utf-8") as f:
        reference = json.load(f)

    rows = []
    for key, ref in reference.items():
        if key not in metrics:
            continue
        if not isinstance(ref, (int, float)) or not isinstance(metrics[key], (int, float)):
            continue

        current = metrics[key]
        if isinstance(ref, int) and isinstance(current, int):
            ok = (current == ref)
        else:
            ok = bool(np.isclose(current, ref, rtol=1e-3, atol=1e-4, equal_nan=True))

        rows.append({
            "metric": key,
            "reference": ref,
            "current_run": current,
            "within_tolerance": ok,
        })

    comparison = pd.DataFrame(rows)
    display(comparison)

    if len(comparison) and comparison["within_tolerance"].all():
        print("Reference check PASSED.")
    else:
        print("Reference check did not fully match. Check software versions/device.")
else:
    print("No reference_metrics.json yet.")
    print("Author: set SAVE_REFERENCE=True once, run this cell, then set it back to False.")

## Completion

In [ ]:
summary_file = ASSET_DIR / "quick_test_run_summary.json"
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("=" * 72)
print("QUICK TEST COMPLETED SUCCESSFULLY")
print("=" * 72)
print("Summary:", summary_file)